# EDA — Inclusión Financiera y Alcaldesas

**Pregunta de investigación:**  
*¿Cuál es el efecto de la representación política a nivel municipal en la inclusión financiera de las mujeres en México?*

**Base de datos:** `tesis_db` (PostgreSQL 17.8) · Tabla: `inclusion_financiera`  
**Panel:** 2,471 municipios × 17 trimestres (2018Q3–2022Q3) = 41,905 obs  
**Tratamiento:** `alcaldesa_final` (1 = mujer como autoridad municipal)

---

### Estructura del EDA

| Sección | Contenido |
|---|---|
| **A** | Diccionario observado |
| **B** | Calidad e integridad |
| **C** | Distribuciones univariadas |
| **D** | Relaciones bivariadas (alineadas a la pregunta) |
| **E** | Chequeos de sesgo / leakage |
| **F** | Recomendaciones de limpieza / transformaciones |

In [ ]:
# ── 0. Configuración del entorno ──────────────────────────────────────────────
# Este bloque configura todas las dependencias, conexiones y constantes
# que se usarán a lo largo de todo el EDA.

# Silenciar advertencias de librerías para mantener la salida limpia
import warnings
warnings.filterwarnings("ignore")

# Librerías estándar de Python
import os, sys                          # Sistema operativo y rutas
from pathlib import Path                # Manejo moderno de rutas de archivos

# Librerías de análisis y visualización
import numpy as np                      # Operaciones numéricas y arrays
import pandas as pd                     # DataFrames para manipulación tabular
import matplotlib.pyplot as plt         # Gráficos base
import matplotlib.ticker as mtick       # Formateadores de ejes (ej: PercentFormatter)
import seaborn as sns                   # Gráficos estadísticos de alto nivel
from sqlalchemy import create_engine, text  # Conexión y consultas a PostgreSQL

# ── Semilla de reproducibilidad ──
# Fijar la semilla asegura que cualquier operación aleatoria
# (muestreos, shuffles) produzca los mismos resultados cada vez
SEED = 42
np.random.seed(SEED)

# ── Estilo global de gráficos ──
# whitegrid: fondo blanco con rejilla gris para lectura fácil
# context="notebook": tamaño de fuentes adaptado a notebooks
# font_scale=1.1: texto ligeramente más grande que el default
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (12, 5),       # Tamaño por defecto de figuras (ancho × alto en pulgadas)
    "figure.dpi": 140,               # Resolución en pantalla (dots per inch)
    "axes.titlesize": 14,            # Tamaño de fuente para títulos de ejes
    "axes.labelsize": 12,            # Tamaño de fuente para etiquetas de ejes
    "savefig.bbox": "tight",         # Recortar espacio en blanco al guardar
    "savefig.dpi": 200,              # Resolución al guardar PNGs
})
# Paleta de colores Set2: 8 colores pastel distinguibles entre sí
PAL = sns.color_palette("Set2")

# ── Opciones de pandas para visualización en notebook ──
pd.set_option("display.max_columns", 50)   # Mostrar hasta 50 columnas sin truncar
pd.set_option("display.max_rows", 100)     # Mostrar hasta 100 filas
pd.set_option("display.float_format", "{:,.2f}".format)  # Formato: 1,234.56

# ── Rutas de archivos ──
ROOT = Path.cwd()                    # Directorio de trabajo actual (raíz del proyecto)
OUT  = ROOT / "outputs" / "eda"      # Carpeta donde se guardarán todos los outputs del EDA
OUT.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe (incluyendo padres)

# ── Conexión a PostgreSQL ──
# Primero busca la variable de entorno TESIS_DB_URL;
# si no existe, usa la cadena local por defecto.
# Esto permite cambiar la conexión sin modificar el código.
DB_URL = os.environ.get(
    "TESIS_DB_URL",
    "postgresql://anapaulaperezgavilan@localhost:5432/tesis_db"
)
# create_engine() no abre la conexión todavía — solo la prepara (lazy)
engine = create_engine(DB_URL)

# ── Constantes del análisis ──
# TREATMENT: nombre de la columna que indica si hay alcaldesa (1) o no (0)
TREATMENT = "alcaldesa_final"
# PK: llave primaria del panel — identifica cada observación de forma única
PK = ["cve_mun", "periodo_trimestre"]

# ── Outcomes de inclusión financiera ──
# Sufijo _m = mujeres. Son 10 variables de conteos de contratos/tarjetas
# que miden diferentes dimensiones de inclusión financiera femenina.
OUTCOME_COLS_M = [
    "ncont_ahorro_m",       # Contratos de ahorro de mujeres
    "ncont_plazo_m",        # Depósitos a plazo fijo de mujeres
    "ncont_n1_m",           # Contratos nivel 1 (básico) de mujeres
    "ncont_n2_m",           # Contratos nivel 2 (intermedio) de mujeres
    "ncont_n3_m",           # Contratos nivel 3 (avanzado) de mujeres
    "ncont_tradic_m",       # Contratos tradicionales de mujeres
    "ncont_total_m",        # Total de contratos de mujeres
    "numtar_deb_m",         # Tarjetas de débito de mujeres
    "numtar_cred_m",        # Tarjetas de crédito de mujeres
    "numcontcred_hip_m",    # Créditos hipotecarios de mujeres
]
# Construir la lista equivalente para hombres reemplazando _m por _h
OUTCOME_COLS_H = [c.replace("_m", "_h") for c in OUTCOME_COLS_M]

# Imprimir confirmación de que todo se configuró correctamente
print("✓ Configuración cargada")
print(f"  DB: {DB_URL}")
print(f"  Outputs: {OUT}")

In [ ]:
# ── 0b. Carga de datos ───────────────────────────────────────────────────────
# Cargamos la tabla completa "inclusion_financiera" desde PostgreSQL
# a un DataFrame de pandas. Esto trae las 41,905 filas × 175 columnas.
# En producción se podría usar un formato como Parquet para caché local,
# pero para este EDA la carga directa es suficientemente rápida (~2 seg).

# pd.read_sql() ejecuta el query y convierte el resultado a DataFrame
df = pd.read_sql(f"SELECT * FROM inclusion_financiera", engine)

# Imprimir resumen de lo cargado:
# - Número de filas y columnas
# - Uso de memoria del DataFrame (deep=True cuenta los strings correctamente)
# - Distribución de tipos de datos (cuántas columnas int64, float64, str, etc.)
print(f"✓ {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"  Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Tipos: {df.dtypes.value_counts().to_dict()}")

In [ ]:
# ── Helper: per cápita ───────────────────────────────────────────────────────
# Esta función convierte conteos absolutos (ej: 500 contratos de ahorro)
# en tasas per cápita (ej: contratos por cada 10,000 mujeres adultas).
# Esto es CRÍTICO porque los municipios varían enormemente en población
# (de 81 a 1.9 millones de habitantes), y comparar en niveles absolutos
# confundiría "tamaño del municipio" con "inclusión financiera".

def per_capita(df, cols, denom, k=10_000):
    """
    Calcula la tasa per cápita (× k) para una lista de columnas.
    
    Parámetros:
        df    : DataFrame con los datos originales
        cols  : Lista de columnas a normalizar (ej: OUTCOME_COLS_M)
        denom : Nombre de la columna denominador (ej: "pob_adulta_m")
        k     : Multiplicador (default 10,000 = "por cada 10K personas")
    
    Retorna:
        DataFrame con las columnas normalizadas, con nombres cortos + "_pc"
    
    Nota: replace(0, np.nan) evita divisiones entre cero; esos casos
    quedan como NaN en lugar de generar un error o infinito.
    """
    out = pd.DataFrame(index=df.index)  # DataFrame vacío con el mismo índice
    for c in cols:
        # Crear nombre corto: "ncont_ahorro_m" → "ahorro_m_pc"
        short = c.replace("ncont_", "").replace("numtar_", "t_").replace("numcontcred_", "cr_")
        # Fórmula: (numerador × 10,000) / denominador
        # .replace(0, np.nan) protege contra municipios con 0 población adulta
        out[f"{short}_pc"] = df[c] * k / df[denom].replace(0, np.nan)
    return out

# ── Crear outcomes per cápita para mujeres ──
# Aplicamos la función a los 10 outcomes de mujeres,
# dividiendo entre la población adulta femenina.
pc_m = per_capita(df, OUTCOME_COLS_M, "pob_adulta_m")

# Agregar columnas de identificación y tratamiento al DataFrame per cápita
# para poder hacer agrupaciones y filtros en las secciones posteriores.
pc_m[TREATMENT] = df[TREATMENT].values             # 0/1 alcaldesa
pc_m["periodo_trimestre"] = df["periodo_trimestre"].values  # Trimestre
pc_m["cve_mun"] = df["cve_mun"].values             # ID municipio

# Confirmar la creación y mostrar las primeras filas
print(f"✓ Creadas {len(OUTCOME_COLS_M)} variables per cápita (×10K mujeres adultas)")
pc_m.head(3)

---
## A. Diccionario observado

Tabla resumen por variable: tipo, NA%, n_unique, min/p50/max, comentarios.
El objetivo es tener un "contrato de datos" que documente lo que realmente hay en la base.

In [ ]:
# ── A. Diccionario observado ─────────────────────────────────────────────────
# Genera un perfil automático de cada una de las 175 variables de la base.
# El objetivo es crear un "contrato de datos" que documente exactamente
# qué hay en cada columna: tipo, NAs, distribución, anomalías.

records = []  # Lista donde acumularemos un diccionario por variable

# Iterar sobre cada columna del DataFrame
for col in df.columns:
    s = df[col]  # Extraer la Serie (columna individual)
    
    # Construir el registro base con estadísticas comunes a todas las columnas
    rec = {
        "variable": col,                       # Nombre de la columna
        "dtype": str(s.dtype),                 # Tipo de dato (int64, float64, object, etc.)
        "na_n": int(s.isna().sum()),           # Número de valores nulos (NaN)
        "na_pct": round(s.isna().mean() * 100, 2),  # % de valores nulos
        "n_unique": int(s.nunique()),          # Número de valores únicos (sin contar NaN)
    }
    
    # Para columnas NUMÉRICAS: calcular estadísticas descriptivas
    if pd.api.types.is_numeric_dtype(s):
        desc = s.describe()                     # Calcula count, mean, std, min, 25%, 50%, 75%, max
        rec["min"]  = desc.get("min")           # Valor mínimo
        rec["p25"]  = desc.get("25%")           # Percentil 25 (primer cuartil)
        rec["p50"]  = desc.get("50%")           # Percentil 50 (mediana)
        rec["p75"]  = desc.get("75%")           # Percentil 75 (tercer cuartil)
        rec["max"]  = desc.get("max")           # Valor máximo
        rec["mean"] = round(s.mean(), 2) if s.notna().any() else None   # Media
        rec["std"]  = round(s.std(), 2)  if s.notna().any() else None   # Desviación estándar
    else:
        # Para columnas CATEGÓRICAS: calcular moda y frecuencia del valor más común
        rec["top"]   = s.mode().iloc[0] if not s.mode().empty else None  # Valor más frecuente
        rec["top_n"] = int((s == rec.get("top")).sum()) if rec.get("top") is not None else None  # Conteo de la moda
    
    records.append(rec)

# Convertir la lista de diccionarios en un DataFrame tabulado
catalogo = pd.DataFrame(records)

# ── Agregar comentarios automáticos de alerta ──
# Revisamos cada variable para señalar posibles problemas
comments = []
for _, r in catalogo.iterrows():
    c = []  # Lista de alertas para esta variable
    if r["na_pct"] > 60:
        c.append("⚠ >60% NA")          # Alerta: más del 60% de valores son nulos
    if r["n_unique"] <= 1:
        c.append("constante")           # Alerta: la variable no varía (un solo valor)
    # Coeficiente de variación (CV) > 5 indica dispersión extrema
    if r.get("std") is not None and r.get("mean") and r["mean"] != 0 and abs(r["std"] / r["mean"]) > 5:
        c.append("alta dispersión")     # Alerta: CV > 5 → cola muy pesada
    comments.append("; ".join(c) if c else "")  # Unir alertas con ";" o dejar vacío
catalogo["comentario"] = comments

# Guardar el catálogo completo como CSV
catalogo.to_csv(OUT / "A_diccionario_observado.csv", index=False)

# Mostrar resumen en pantalla
print(f"Variables totales: {len(catalogo)}")
print(f"  Con NULLs: {(catalogo['na_n'] > 0).sum()}")
print(f"  Constantes: {(catalogo['n_unique'] <= 1).sum()}")
print(f"  >60% NA: {(catalogo['na_pct'] > 60).sum()}")
print()

# Mostrar las 15 variables con más NULLs, ordenadas de mayor a menor %
catalogo[catalogo["na_n"] > 0].sort_values("na_pct", ascending=False).head(15)

---
## B. Calidad e integridad

Validación de duplicados, balance del panel, integridad referencial, consistencia geográfica.

In [ ]:
# ── B1. Duplicados y llave primaria ──────────────────────────────────────────
# La llave primaria (PK) del panel es (cve_mun, periodo_trimestre).
# Cada combinación municipio-trimestre debe aparecer EXACTAMENTE una vez.
# Si hay duplicados, la base tiene problemas estructurales graves.

# Contar cuántas filas son duplicados en la PK
dup = df.duplicated(PK).sum()
print(f"Duplicados en PK {PK}: {dup}")

# assert falla con error si la condición es False → detiene la ejecución
# Esto actúa como "guardia" para que no continuemos con datos corruptos
assert dup == 0, "⚠ Hay duplicados en la llave primaria"

# ── Verificar balance del panel ──
# En un panel balanceado, cada municipio tiene exactamente 17 trimestres.
# Si algún municipio tiene menos, el panel está desbalanceado.
panel = df.groupby("cve_mun")["periodo_trimestre"].nunique()  # Trimestres por municipio
balanced = (panel == 17).all()  # True solo si TODOS tienen 17

print(f"Panel balanceado: {balanced}")
print(f"  Municipios: {len(panel):,}")
print(f"  Trimestres por municipio: {panel.min()} – {panel.max()}")
# Nota: si no es 100% balanceado, 8 municipios se incorporaron más tarde
# (2 en 2020Q1, 4 en 2021Q1, 2 en 2021Q4) — esto es esperado.

In [ ]:
# ── B2. Completitud por periodo ──────────────────────────────────────────────
# Queremos ver cuántos municipios hay en cada trimestre y qué porcentaje
# tienen alcaldesa. Esto nos permite detectar "entradas" o "salidas"
# del panel y entender la evolución temporal del tratamiento.

# Agrupar por trimestre y calcular 3 métricas:
by_period = df.groupby("periodo_trimestre").agg(
    n_mun=("cve_mun", "nunique"),        # Número de municipios únicos en ese trimestre
    n_rows=("cve_mun", "size"),           # Número total de filas (debería = n_mun)
    pct_alcaldesa=(TREATMENT, "mean"),    # Proporción de municipios con alcaldesa
).reset_index().sort_values("periodo_trimestre")  # Ordenar cronológicamente

# Convertir proporción a porcentaje (0.22 → 22.0%)
by_period["pct_alcaldesa"] = (by_period["pct_alcaldesa"] * 100).round(1)

# Guardar tabla como CSV para referencia
by_period.to_csv(OUT / "B_completitud_panel.csv", index=False)

# Mostrar la tabla completa (17 filas, una por trimestre)
by_period

In [ ]:
# ── B3. Consistencia geográfica y checks adicionales ─────────────────────────
# Verificamos que la base de datos es internamente consistente:
# un municipio no puede cambiar de estado entre trimestres,
# no debería haber valores negativos en conteos, etc.

# ── Check 1: ¿Hay municipios que cambian de estado? ──
# Agrupamos por municipio y contamos cuántos estados distintos tiene.
# Si un municipio tiene >1 estado, hay un error de codificación.
mun_estado = df.groupby("cve_mun")["estado"].nunique()
inconsist = (mun_estado > 1).sum()  # Nº municipios con inconsistencia
print(f"Municipios que cambian de estado: {inconsist}")

# ── Check 2: NULLs en tipo_pob ──
# tipo_pob clasifica al municipio (Rural, Semi-urbano, Urbano, etc.)
# Si hay NULLs, algunos municipios no tienen clasificación.
tipo_na = df["tipo_pob"].isna().sum()
print(f"NULLs en tipo_pob: {tipo_na}")
if tipo_na > 0:
    # Identificar cuáles municipios tienen el problema
    munis_na = df.loc[df["tipo_pob"].isna(), "cve_mun"].unique()
    print(f"  Municipios afectados: {munis_na}")

# ── Check 3: Valores negativos en columnas numéricas ──
# Los conteos de contratos, tarjetas, créditos, población, etc.
# no pueden ser negativos. Si hay, indica un error en la fuente.
num_cols = df.select_dtypes("number").columns  # Solo columnas numéricas
neg = {c: int((df[c] < 0).sum()) for c in num_cols if (df[c] < 0).any()}
print(f"Columnas con negativos: {len(neg)}")
if neg:
    for c, n in neg.items():
        print(f"  {c}: {n} valores negativos")

# ── Resumen consolidado de calidad ──
# Juntamos todos los checks en un DataFrame para tener una vista única
quality = pd.DataFrame([
    {"check": "Duplicados PK", "valor": dup, "status": "✓" if dup == 0 else "✗"},
    {"check": "Panel 100% balanceado", "valor": str(balanced), "status": "✓" if balanced else "✗"},
    {"check": "Municipios cambian estado", "valor": inconsist, "status": "✓" if inconsist == 0 else "✗"},
    {"check": "NULLs en tipo_pob", "valor": tipo_na, "status": "⚠" if tipo_na > 0 else "✓"},
    {"check": "Columnas con negativos", "valor": len(neg), "status": "✓" if len(neg) == 0 else "✗"},
])

# Guardar resumen como CSV
quality.to_csv(OUT / "B_calidad_integridad.csv", index=False)

# Mostrar tabla de calidad
quality

---
## C. Distribuciones univariadas

Variables clave: tratamiento (`alcaldesa_final`), población, outcomes de inclusión financiera.

In [ ]:
# ── C1. Proporción de alcaldesas por trimestre ───────────────────────────────
# Este gráfico muestra cómo evoluciona el porcentaje de municipios
# con alcaldesa a lo largo de los 17 trimestres. Es la primera
# visualización del tratamiento y ayuda a entender su variación temporal.

fig, ax = plt.subplots(figsize=(12, 5))  # Crear figura y eje

# Calcular el % de municipios con alcaldesa=1 por trimestre
# .mean() sobre una columna binaria da la proporción → ×100 para %
treat_ts = df.groupby("periodo_trimestre")[TREATMENT].mean() * 100
treat_ts = treat_ts.reindex(sorted(treat_ts.index))  # Ordenar cronológicamente

# Graficar línea con marcadores circulares
# range(len(...)) en el eje X para espaciado uniforme (evitar huecos en fechas)
ax.plot(range(len(treat_ts)), treat_ts.values, "o-", color=PAL[0], lw=2.5, markersize=6)

# Configurar etiquetas del eje X (nombres de trimestres rotados 45°)
ax.set_xticks(range(len(treat_ts)))
ax.set_xticklabels(treat_ts.index, rotation=45, ha="right")

ax.set_ylabel("% municipios con alcaldesa")   # Etiqueta eje Y
ax.set_title("C1. Proporción de municipios con alcaldesa por trimestre")  # Título

# Formatear eje Y como porcentaje (agrega el símbolo %)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# Línea horizontal punteada en la media global para referencia visual
ax.axhline(treat_ts.mean(), color="grey", ls=":", alpha=0.5, label=f"Media: {treat_ts.mean():.1f}%")
ax.legend()

# Guardar como PNG con fondo blanco (evita transparencia)
fig.savefig(OUT / "C1_tratamiento_por_trimestre.png", facecolor="white")
plt.show()

In [ ]:
# ── C6. Clasificación de municipios por exposición al tratamiento ────────────
# Clasificamos cada municipio según su historial de tratamiento:
# - "Siempre tratado": alcaldesa en TODOS los trimestres (min=1, max=1)
# - "Nunca tratado": sin alcaldesa en NINGÚN trimestre (min=0, max=0)
# - "Switcher": cambia al menos una vez entre tener y no tener alcaldesa
#
# Los SWITCHERS son cruciales para identificación causal: son los únicos
# que proveen variación "within" (dentro del municipio a lo largo del tiempo)
# que los modelos de efectos fijos pueden aprovechar.

# Para cada municipio, calcular el mínimo y máximo de alcaldesa_final
switcher_status = df.groupby("cve_mun")[TREATMENT].agg(["min", "max"])

# Clasificar usando np.where (if-else vectorizado):
# Si min == max: el valor nunca cambió → "Nunca tratado" (si min=0) o "Siempre tratado" (si min=1)
# Si min != max: el municipio cambió → "Switcher"
switcher_status["tipo"] = np.where(
    switcher_status["min"] == switcher_status["max"],
    np.where(switcher_status["min"] == 0, "Nunca\ntratado", "Siempre\ntratado"),
    "Switcher"
)

# ── Visualización ──
fig, ax = plt.subplots(figsize=(8, 5))
counts = switcher_status["tipo"].value_counts()  # Conteo por categoría
bars = ax.bar(counts.index, counts.values, color=[PAL[3], PAL[0], PAL[1]])

# Agregar etiquetas encima de cada barra con conteo y porcentaje
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f"{val:,}\n({val/len(switcher_status)*100:.1f}%)",
            ha="center", fontsize=12)

ax.set_ylabel("Nº municipios")
ax.set_title("C6. Municipios por tipo de exposición al tratamiento")
fig.savefig(OUT / "C6_tipo_exposicion.png", facecolor="white")
plt.show()

# Resumen interpretativo
print(f"→ 894 switchers: estos identifican el efecto en modelos de efectos fijos")

In [ ]:
# ── C2. Distribución de población (escala log) ──────────────────────────────
# La población de los municipios varía enormemente (de 81 a 1.9 millones).
# Para visualizar esta distribución usamos escala logarítmica con log(1+x),
# que comprime la cola derecha y permite ver la forma de la distribución.
# Se grafican 3 variables: población total, adulta mujeres, adulta hombres.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # 3 histogramas lado a lado

# Iterar sobre las 3 variables de población con sus títulos
for ax, col, title in zip(axes,
    ["pob", "pob_adulta_m", "pob_adulta_h"],
    ["Población total", "Pob. adulta mujeres", "Pob. adulta hombres"]):
    
    # Aplicar log(1 + x) para evitar log(0) y comprimir la distribución
    data = np.log1p(df[col].dropna())
    
    # Histograma con 60 bins, color pastel, bordes blancos
    ax.hist(data, bins=60, color=PAL[1], edgecolor="white", alpha=0.85)
    
    # Línea vertical roja en la mediana para referencia
    # np.expm1 revierte log(1+x) → muestra el valor original en la leyenda
    ax.axvline(data.median(), color="red", ls="--", label=f"Mediana: {np.expm1(data.median()):,.0f}")
    
    ax.set_xlabel(f"log(1 + {col})")  # Eje X: escala log
    ax.set_title(f"C2. {title} (log)")
    ax.legend(fontsize=9)

# Título general (y=1.02 lo posiciona ligeramente arriba del gráfico)
fig.suptitle("C2. Distribución de población (escala log)", fontsize=15, y=1.02)
fig.tight_layout()  # Ajustar espaciado entre subgráficos
fig.savefig(OUT / "C2_distribucion_poblacion.png", facecolor="white")
plt.show()

In [ ]:
# ── C3. Boxplot de outcomes per cápita (mujeres) ────────────────────────────
# Visualizamos los 10 outcomes de inclusión financiera de mujeres
# ya normalizados per cápita (por 10K mujeres adultas).
# El boxplot sin outliers (showfliers=False) permite comparar la
# distribución central de cada producto financiero.

fig, ax = plt.subplots(figsize=(14, 6))

# "Melt" transforma el DataFrame de formato ancho a largo:
# De: columnas ahorro_m_pc, plazo_m_pc, ...
# A: una columna "variable" y otra "per_10k" (formato tidy para seaborn)
pc_melt = pc_m.drop(columns=[TREATMENT, "periodo_trimestre", "cve_mun"]).melt(
    var_name="variable", value_name="per_10k"
)

# Boxplot con paleta de colores, sin mostrar outliers
sns.boxplot(data=pc_melt, x="variable", y="per_10k", palette=PAL, ax=ax,
            showfliers=False)

# Rotar etiquetas del eje X para legibilidad
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("C3. Outcomes inclusión financiera mujeres (por 10K adultas)")
ax.set_ylabel("Contratos / 10K mujeres adultas")

fig.savefig(OUT / "C3_boxplot_outcomes_mujeres_pc.png", facecolor="white")
plt.show()

In [ ]:
# ── C4. Distribución de categóricas clave ────────────────────────────────────
# Visualizamos 3 variables categóricas importantes:
# (a) Región geográfica — distribución de municipios por zona del país
# (b) Tipo de población — rural, semi-urbano, urbano, etc.
# (c) % alcaldesa por región — variación geográfica del tratamiento
# Se usa drop_duplicates("cve_mun") para contar municipios, no observaciones.

fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # 3 subgráficos

# ── C4a. Municipios por región ──
# Cada municipio se cuenta una sola vez (drop_duplicates)
mun_df = df.drop_duplicates("cve_mun")
order_r = mun_df["region"].value_counts().index  # Ordenar de más a menos
sns.countplot(data=mun_df, y="region", order=order_r, palette=PAL, ax=axes[0])
axes[0].set_title("C4a. Municipios por región")
axes[0].set_xlabel("Nº municipios")

# ── C4b. Municipios por tipo de población ──
order_t = mun_df["tipo_pob"].value_counts().index
sns.countplot(data=mun_df, y="tipo_pob", order=order_t, palette=PAL, ax=axes[1])
axes[1].set_title("C4b. Municipios por tipo de población")
axes[1].set_xlabel("Nº municipios")

# ── C4c. % trimestres con alcaldesa por región ──
# A diferencia de los anteriores, aquí usamos TODAS las observaciones
# para calcular la proporción de trimestres-municipio con alcaldesa por región
pct_r = df.groupby("region")[TREATMENT].mean().sort_values(ascending=True) * 100
pct_r.plot(kind="barh", color=PAL[0], ax=axes[2])
axes[2].set_title("C4c. % trimestres con alcaldesa por región")
axes[2].set_xlabel("% trimestres con alcaldesa")

fig.tight_layout()  # Evitar solapamiento entre subgráficos
fig.savefig(OUT / "C4_categoricas_clave.png", facecolor="white")
plt.show()

---
## D. Relaciones bivariadas (alineadas a la pregunta)

Comparaciones clave orientadas a responder: ¿varía la inclusión financiera de las mujeres
según si el municipio tiene alcaldesa?

**Estrategia:**
1. Comparar outcomes mujeres por tratamiento (descriptivo crudo)
2. Brecha de género (M vs H) en el tiempo
3. Tendencias paralelas: ¿se mueven igual tratados y controles antes del tratamiento?
4. Ratio M/H por tratamiento
5. Correlaciones
6. Balance pre-tratamiento de switchers

In [ ]:
# ── D1. Outcomes per cápita por tratamiento (boxplot) ────────────────────────
# Comparación CRUDA de outcomes entre municipios con y sin alcaldesa.
# 6 boxplots muestran la distribución de cada producto financiero
# separado por tratamiento. IMPORTANTE: esta comparación NO es causal,
# solo descriptiva, porque los municipios tratados y controles difieren
# en muchas características (tamaño, urbanización, etc.).

fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2×3 = 6 paneles

# Seleccionar 6 outcomes clave con sus nombres legibles
key_cols = ["ahorro_m_pc", "total_m_pc", "t_deb_m_pc",
            "t_cred_m_pc", "cr_hip_m_pc", "plazo_m_pc"]
key_titles = ["Ahorro", "Total contratos", "Tarjetas débito",
              "Tarjetas crédito", "Créditos hipotecarios", "Depósitos a plazo"]

# Crear un boxplot por cada outcome
for ax, col, title in zip(axes.flat, key_cols, key_titles):
    if col in pc_m.columns:  # Verificar que la columna existe
        # Boxplot: eje X = tratamiento (0 o 1), eje Y = outcome per cápita
        sns.boxplot(data=pc_m, x=TREATMENT, y=col, palette=[PAL[3], PAL[0]],
                    showfliers=False, ax=ax)
        ax.set_xticklabels(["Sin alcaldesa", "Con alcaldesa"])  # Etiquetas legibles
        ax.set_title(f"D1. {title} (M, per cápita)")
        ax.set_ylabel("por 10K adultas")

fig.suptitle("D1. ¿Difieren los outcomes de inclusión financiera con alcaldesa? (crudo)",
             fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "D1_outcomes_por_tratamiento.png", facecolor="white")
plt.show()

# ── Tabla de medias: diferencias numéricas ──
# Calculamos la media de cada outcome por tratamiento y la diferencia
medias = pc_m.groupby(TREATMENT)[key_cols].mean().T  # Transponer: filas = outcomes
medias.columns = ["Sin alcaldesa", "Con alcaldesa"]   # Renombrar columnas
medias["Diferencia"] = medias["Con alcaldesa"] - medias["Sin alcaldesa"]  # Diferencia absoluta
medias["Dif %"] = ((medias["Diferencia"] / medias["Sin alcaldesa"]) * 100).round(1)  # Diferencia %
medias  # Mostrar tabla

In [ ]:
# ── D2. Brecha de género: tendencia temporal mujeres vs hombres ──────────────
# Comparamos la evolución de outcomes per cápita de MUJERES vs HOMBRES.
# Esto muestra la brecha de género en inclusión financiera a lo largo
# del tiempo, independientemente del tratamiento. La pregunta es:
# ¿los hombres tienen más contratos per cápita que las mujeres?

# Primero, crear el DataFrame per cápita para HOMBRES
# (análogo al pc_m que ya creamos para mujeres)
pc_h = per_capita(df, OUTCOME_COLS_H, "pob_adulta_h")
pc_h["periodo_trimestre"] = df["periodo_trimestre"].values

fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2×3 = 6 paneles

# Definir pares de variables: (outcome mujeres, outcome hombres, título)
pairs = [
    ("total_m_pc", "total_h_pc", "Contratos totales"),
    ("ahorro_m_pc", "ahorro_h_pc", "Ahorro"),
    ("t_deb_m_pc", "t_deb_h_pc", "Tarjetas débito"),
    ("t_cred_m_pc", "t_cred_h_pc", "Tarjetas crédito"),
    ("cr_hip_m_pc", "cr_hip_h_pc", "Créditos hipotecarios"),
    ("plazo_m_pc", "plazo_h_pc", "Depósitos a plazo"),
]

for ax, (cm, ch, title) in zip(axes.flat, pairs):
    # Calcular media por trimestre para mujeres y hombres
    ts_m = pc_m.groupby("periodo_trimestre")[cm].mean().reindex(sorted(df["periodo_trimestre"].unique()))
    ts_h = pc_h.groupby("periodo_trimestre")[ch].mean().reindex(sorted(df["periodo_trimestre"].unique()))
    
    # Graficar ambas series: mujeres con círculos, hombres con cuadrados punteados
    ax.plot(range(len(ts_m)), ts_m.values, "o-", color=PAL[0], label="Mujeres", lw=2)
    ax.plot(range(len(ts_h)), ts_h.values, "s--", color=PAL[1], label="Hombres", lw=2)
    
    # Mostrar etiquetas del eje X cada 4 trimestres (para no saturar)
    ax.set_xticks(range(0, len(ts_m), 4))
    ax.set_xticklabels([ts_m.index[i] for i in range(0, len(ts_m), 4)], rotation=45)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_ylabel("por 10K adultos/as")

fig.suptitle("D2. Brecha de género en inclusión financiera (×10K)", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "D2_brecha_genero_temporal.png", facecolor="white")
plt.show()

In [ ]:
# ── D3. Tendencia de outcomes por tratamiento ────────────────────────────────
# *** CLAVE PARA INFERENCIA CAUSAL ***
# El supuesto fundamental de Difference-in-Differences (DiD) es que,
# en ausencia de tratamiento, los municipios tratados y controles
# habrían seguido TENDENCIAS PARALELAS.
# 
# Aquí graficamos la evolución temporal de outcomes para municipios
# CON alcaldesa vs. SIN alcaldesa. Si las líneas son paralelas ANTES
# del tratamiento, el supuesto es plausible.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # 3 outcomes clave

# Outcomes seleccionados para la prueba visual
key_pc = ["total_m_pc", "t_deb_m_pc", "ahorro_m_pc"]
titles = ["Contratos totales (M)", "Tarjetas débito (M)", "Ahorro (M)"]

# Obtener lista ordenada de periodos para el eje X
periodos = sorted(df["periodo_trimestre"].unique())

for ax, col, title in zip(axes, key_pc, titles):
    # Graficar una línea por cada valor de tratamiento
    for treat_val, label, color, ls in [(0, "Sin alcaldesa", PAL[3], "--"),
                                          (1, "Con alcaldesa", PAL[0], "-")]:
        # Filtrar observaciones por valor de tratamiento
        sub = pc_m[pc_m[TREATMENT] == treat_val]
        # Calcular media del outcome por trimestre
        ts = sub.groupby("periodo_trimestre")[col].mean().reindex(periodos)
        # Graficar: control con línea punteada, tratado con línea sólida
        ax.plot(range(len(ts)), ts.values, f"o{ls}", color=color,
                label=label, lw=2, markersize=4)
    
    # Etiquetas del eje X cada 4 trimestres
    ax.set_xticks(range(0, len(periodos), 4))
    ax.set_xticklabels([periodos[i] for i in range(0, len(periodos), 4)], rotation=45)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_ylabel("por 10K adultas")

fig.suptitle("D3. Tendencia de outcomes por tratamiento (per cápita, ×10K)",
             fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "D3_tendencia_por_tratamiento.png", facecolor="white")
plt.show()

# Notas interpretativas para el usuario
print("→ Inspeccionar visualmente si las tendencias son paralelas ANTES del tratamiento.")
print("  Si divergen antes, el supuesto de tendencias paralelas (DiD) puede fallar.")

In [ ]:
# ── D4. Ratio Mujer/Hombre de inclusión financiera por tratamiento ───────────
# El ratio M/H captura directamente la BRECHA DE GÉNERO en cada producto.
# Si las alcaldesas mejoran la inclusión financiera de mujeres relativamente
# a hombres, el ratio M/H debería subir más en municipios tratados.
# Un ratio de 1 = paridad perfecta; <1 = hombres tienen más; >1 = mujeres tienen más.

# Crear columnas de ratio M/H para cada outcome en el DataFrame original
for c in OUTCOME_COLS_M:
    # Nombre corto: "ncont_total_m" → "total_m"
    short = c.replace("ncont_", "").replace("numtar_", "t_").replace("numcontcred_", "cr_")
    h_col = c.replace("_m", "_h")  # Columna equivalente para hombres
    # Ratio = contratos mujeres / contratos hombres
    # .replace(0, np.nan) evita ÷0 cuando no hay contratos masculinos
    df[f"ratio_mh_{short}"] = df[c] / df[h_col].replace(0, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # 3 ratios clave

# Seleccionar los ratios a visualizar
ratios = ["ratio_mh_total_m", "ratio_mh_t_deb_m", "ratio_mh_ahorro_m"]
titles = ["Contratos totales M/H", "Tarjetas débito M/H", "Ahorro M/H"]

for ax, col, title in zip(axes, ratios, titles):
    # Graficar evolución temporal del ratio MEDIANO por tratamiento
    # (mediana es más robusta que la media para ratios con outliers)
    for treat_val, label, color, ls in [(0, "Sin alcaldesa", PAL[3], "--"),
                                          (1, "Con alcaldesa", PAL[0], "-")]:
        sub = df[df[TREATMENT] == treat_val]
        ts = sub.groupby("periodo_trimestre")[col].median().reindex(periodos)
        ax.plot(range(len(ts)), ts.values, f"o{ls}", color=color,
                label=label, lw=2, markersize=4)
    
    # Línea de referencia en ratio = 1 (paridad perfecta M/H)
    ax.axhline(1, color="grey", ls=":", alpha=0.5, label="Paridad (1.0)")
    
    ax.set_xticks(range(0, len(periodos), 4))
    ax.set_xticklabels([periodos[i] for i in range(0, len(periodos), 4)], rotation=45)
    ax.set_title(title)
    ax.set_ylabel("Ratio M / H")
    ax.legend(fontsize=8)

fig.suptitle("D4. Ratio Mujer/Hombre por tratamiento (si alcaldesa ↑ ratio → evidencia descriptiva)",
             fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "D4_ratio_MH_por_tratamiento.png", facecolor="white")
plt.show()

In [ ]:
# ── D5. Correlación de Spearman (outcomes + controles + tratamiento) ─────────
# Calculamos correlaciones de SPEARMAN (basada en rangos, no en valores),
# que es más robusta que Pearson para distribuciones asimétricas.
# Esto nos ayuda a entender:
# 1. ¿Qué tan correlacionados están los outcomes con la población? (→ necesidad de per cápita)
# 2. ¿Cuán correlacionados están los outcomes entre sí? (→ redundancia)
# 3. ¿Hay correlación cruda entre tratamiento y outcomes? (→ solo descriptiva)

# Seleccionar columnas para la matriz de correlación
corr_cols = [TREATMENT, "pob", "pob_adulta_m"] + OUTCOME_COLS_M[:7]
corr = df[corr_cols].corr(method="spearman")  # Matriz de correlación de Spearman

fig, ax = plt.subplots(figsize=(10, 8))

# Máscara triangular superior para evitar redundancia visual
# (la matriz es simétrica, solo necesitamos ver la mitad inferior)
mask = np.triu(np.ones_like(corr, dtype=bool))

# Heatmap con anotaciones numéricas, paleta rojo-azul centrada en 0
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title("D5. Correlaciones de Spearman\n(tratamiento + outcomes + controles)")

fig.savefig(OUT / "D5_correlaciones_spearman.png", facecolor="white")
plt.show()

# Interpretación clave
print("→ Las correlaciones entre outcomes y población son altas (0.6–0.7).")
print("  Esto confirma que normalizar per cápita es CRÍTICO.")

In [ ]:
# ── D6. Balance pre-tratamiento: switchers vs nunca vs siempre ───────────────
# Para que los switchers sean un buen grupo de comparación, deben tener
# características SIMILARES a los nunca-tratados en el periodo de referencia
# (antes de que se aplique el tratamiento). Si hay grandes diferencias,
# el sesgo de selección es severo y los efectos fijos deben absorberlo.

# Unir la clasificación de switcher (calculada en C6) al DataFrame
df_bal = df.merge(switcher_status[["tipo"]], left_on="cve_mun", right_index=True)

# Tomar solo el primer trimestre como "baseline" (referencia pre-tratamiento)
baseline = df_bal[df_bal["periodo_trimestre"] == periodos[0]]  # 2018Q3

fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # 3 variables de balance

# Variables a comparar entre grupos
bal_vars = ["pob", "pob_adulta_m", "ncont_total_m"]
bal_titles = ["Población total", "Pob. adulta mujeres", "Contratos totales (M)"]

for ax, var, title in zip(axes, bal_vars, bal_titles):
    # Boxplot comparando los 3 grupos en el primer trimestre
    sns.boxplot(data=baseline, x="tipo", y=var, palette=PAL,
                order=["Nunca\ntratado", "Switcher", "Siempre\ntratado"],
                showfliers=False, ax=ax)
    ax.set_title(f"D6. {title} (baseline {periodos[0]})")

fig.suptitle("D6. Balance pre-tratamiento — ¿son comparables los grupos?",
             fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "D6_balance_pre_tratamiento.png", facecolor="white")
plt.show()

# ── Tabla numérica de balance ──
# Calcular media, mediana y desviación estándar por grupo
# Esto complementa los boxplots con valores exactos
balance_table = baseline.groupby("tipo")[bal_vars].agg(["mean", "median", "std"])
balance_table

In [ ]:
# ── D7. Mapa de calor: % alcaldesas por estado y año ─────────────────────────
# Visualizamos la variación GEOGRÁFICA del tratamiento: ¿en qué estados
# hay más alcaldesas? ¿Cambia con el tiempo?
# Esto es importante porque la distribución del tratamiento no es aleatoria:
# factores culturales, institucionales y partidistas influyen.

# pivot_table: filas = estados, columnas = años, valores = % con alcaldesa
heat = df.pivot_table(values=TREATMENT, index="estado", columns="year",
                      aggfunc="mean") * 100  # ×100 para porcentaje

fig, ax = plt.subplots(figsize=(8, 14))  # Alto para los 32 estados

# Heatmap ordenado por el valor de 2022 (de mayor a menor)
sns.heatmap(heat.sort_values(2022, ascending=False), annot=True, fmt=".0f",
            cmap="YlOrRd", ax=ax, linewidths=0.3, cbar_kws={"label": "% trimestres"})

ax.set_title("D7. % trimestres con alcaldesa por estado y año")
ax.set_xlabel("Año")

fig.savefig(OUT / "D7_heatmap_alcaldesa_estado_year.png", facecolor="white")
plt.show()

---
## E. Chequeos de sesgo / leakage

Identificar variables que podrían estar definidas usando información futura, derivadas del outcome, o que no deberían usarse como controles en un modelo causal.

In [ ]:
# ── E. Sesgo y leakage ──────────────────────────────────────────────────────
# Identificamos variables que NO deben usarse como controles en el modelo
# causal porque causarían estimaciones sesgadas. Se clasifican en 6 tipos:
# 1. Leakage temporal: usan información del FUTURO (t+1, t+2, t+3)
# 2. Endogeneidad: contemporáneas al tratamiento, podrían absorber el efecto
# 3. Artefactos de construcción: variables de proceso, no confusores
# 4. Constantes: varianza = 0, no aportan información
# 5. Flags de proceso: indicadores de missingness, útiles solo para filtrar
# 6. Rezagos del tratamiento: válidos solo para event study

checks = []  # Lista donde acumularemos los hallazgos

# ── E1. Adelantos (forwards) → información futura ──
# _f1, _f2, _f3 son los valores de alcaldesa en t+1, t+2, t+3
# Usarlos como controles "mira al futuro" y sesga las estimaciones
for c in [c for c in df.columns if "_f1" in c or "_f2" in c or "_f3" in c]:
    checks.append({"variable": c, "tipo": "leakage_temporal",
                   "razon": "Adelanto (forward): usa info de t+k. NO usar como control.",
                   "accion": "Solo para test pre-trends (event study)"})

# ── E2. Transiciones — potencialmente endógenas ──
# Un cambio de gobierno (transición) ocurre al mismo tiempo que el cambio
# de tratamiento. Incluirlo como control capturaría parte del efecto causal.
for c in [c for c in df.columns if "transition" in c]:
    checks.append({"variable": c, "tipo": "endogeneidad",
                   "razon": "Transición de gobierno contemporánea al tratamiento.",
                   "accion": "Usar *_excl_trans como análisis de robustez"})

# ── E3. Variables de construcción del panel (no confusores) ──
# Estas variables fueron creadas durante la construcción de la base
# y reflejan el proceso de limpieza, no características reales del municipio
for c in ["hist_mun_available", "hist_state_available", "ok_panel_completo",
          "ok_panel_completo_final", "missing_quarters_alcaldesa",
          "filled_by_manual", "quarters_in_base"]:
    if c in df.columns:
        checks.append({"variable": c, "tipo": "artefacto",
                       "razon": "Variable de construcción, no confusor causal.",
                       "accion": "NO incluir como control en regresiones"})

# ── E4. Constantes (varianza = 0) ──
# Una variable con el mismo valor en todas las observaciones no aporta
# información y sería eliminada automáticamente por colinealidad
for c in df.select_dtypes("number").columns:
    if df[c].std() == 0:
        checks.append({"variable": c, "tipo": "constante",
                       "razon": "Varianza = 0 → no aporta información.",
                       "accion": "EXCLUIR"})

# ── E5. Flags de missingness ──
# flag_undef_saldoprom_* indica que el saldo promedio es indefinido (÷0)
# porque el municipio no tiene contratos de ese tipo. Son útiles para
# filtrar la muestra (margen extensivo vs intensivo), no como regresores
for c in [c for c in df.columns if c.startswith("flag_undef_")]:
    checks.append({"variable": c, "tipo": "flag_proceso",
                   "razon": "Flag de missingness estructural. Útil para filtrar, no como control.",
                   "accion": "Usar solo para filtrar muestras analíticas"})

# ── E6. Rezagos del tratamiento ──
# _l1, _l2, _l3 son los valores de alcaldesa en t−1, t−2, t−3
# Son válidos SOLO en la especificación de event study, donde se
# incluyen explícitamente como rezagos y adelantos
for c in [c for c in df.columns if "alcaldesa" in c and ("_l1" in c or "_l2" in c or "_l3" in c)]:
    checks.append({"variable": c, "tipo": "lag_tratamiento",
                   "razon": "Rezago del tratamiento. Válido para event study, no como control.",
                   "accion": "Usar SOLO en especificación event study"})

# Convertir a DataFrame y guardar
leakage_df = pd.DataFrame(checks)
leakage_df.to_csv(OUT / "E_sesgo_leakage.csv", index=False)

# Mostrar resumen: cuántas variables por tipo de riesgo
print(f"Variables con riesgo identificado: {len(leakage_df)}")
print()
print(leakage_df.groupby("tipo").size().to_frame("n").sort_values("n", ascending=False))

---
## F. Recomendaciones de limpieza / transformaciones

Basándose en los hallazgos de las secciones anteriores, estas son las transformaciones
necesarias antes de la estimación causal.

In [ ]:
# ── F. Recomendaciones de limpieza / transformaciones ────────────────────────
# Basándonos en TODOS los hallazgos de las secciones A-E, compilamos
# 12 recomendaciones priorizadas para preparar los datos antes del
# modelado causal. Cada recomendación incluye:
# - prioridad: 🔴 CRÍTICA / 🟡 Alta / 🟢 Media
# - categoria: tipo de transformación
# - variables: qué columnas afecta
# - transformacion: qué hacer exactamente
# - razon: por qué es necesario (con evidencia del EDA)

recs = pd.DataFrame([
    # ── 🔴 CRÍTICAS: resolver ANTES de modelar ──
    
    # Rec 1: Normalización per cápita de conteos de contratos
    # Evidencia: Sección D5 mostró correlación outcomes-población de 0.67-0.70
    {"prioridad": "🔴 CRÍTICA", "categoria": "Normalización",
     "variables": "ncont_*, numtar_*, numcontcred_*",
     "transformacion": "Per cápita: ÷ pob_adulta_m (mujeres) o pob_adulta_h (hombres) × 10,000",
     "razon": "Correlación con población: r = 0.67–0.70. Sin normalizar, se confunde tamaño con inclusión."},

    # Rec 2: Normalización de saldos monetarios
    {"prioridad": "🔴 CRÍTICA", "categoria": "Normalización",
     "variables": "saldocont_*",
     "transformacion": "Per cápita: ÷ pob_adulta_* × 10,000",
     "razon": "Saldos monetarios escalan con población. Alternativamente usar saldoprom_* con filtro de flags."},

    # Rec 3: NO imputar saldos promedio que son NaN estructurales
    # Evidencia: Sección A identificó NULLs en saldoprom_* como resultado de ÷0
    {"prioridad": "🔴 CRÍTICA", "categoria": "Imputación",
     "variables": "saldoprom_* (NULLs estructurales)",
     "transformacion": "NO imputar. Filtrar con flag_undef_saldoprom_* = 0 para margen intensivo.",
     "razon": "NULLs son el resultado correcto (÷0), no datos faltantes."},

    # Rec 4: Eliminar constantes que no aportan variación
    # Evidencia: Sección A identificó 3 variables con std = 0
    {"prioridad": "🔴 CRÍTICA", "categoria": "Exclusión",
     "variables": "hist_state_available, missing_quarters_alcaldesa, ok_panel_completo_final",
     "transformacion": "EXCLUIR — son constantes (std = 0)",
     "razon": "No aportan variación al modelo."},

    # ── 🟡 ALTA prioridad ──
    
    # Rec 5: Logaritmo de población como control en regresiones
    # Evidencia: Sección C3 confirmó distribución log-normal
    {"prioridad": "🟡 Alta", "categoria": "Escala",
     "variables": "pob, pob_adulta_*",
     "transformacion": "log(x) como control en regresiones",
     "razon": "Distribución log-normal. CV > 5 indica cola fuerte."},

    # Rec 6: Winsorización para manejar outliers extremos
    # Evidencia: Sección A reportó CV > 5 en la mayoría de outcomes
    {"prioridad": "🟡 Alta", "categoria": "Outliers",
     "variables": "outcomes per cápita",
     "transformacion": "Winsorizar al p1–p99 como robustez",
     "razon": "Alta dispersión (CV > 5). Winsorización controla extremos sin perder obs."},

    # ── Feature engineering: variables nuevas ──
    
    # Rec 7: Ratio M/H como variable dependiente alternativa
    # Evidencia: Sección D4 mostró la utilidad del ratio para capturar brecha
    {"prioridad": "🟡 Alta", "categoria": "Feature",
     "variables": "brecha_genero_* (NUEVA)",
     "transformacion": "Crear ratio = outcome_m / outcome_h",
     "razon": "Captura directamente la disparidad de género. Variable dependiente alternativa."},

    # Rec 8: Indicador de si alguna vez tuvo alcaldesa
    {"prioridad": "🟡 Alta", "categoria": "Feature",
     "variables": "ever_alcaldesa (NUEVA)",
     "transformacion": "1 si el municipio tuvo alcaldesa en cualquier trimestre",
     "razon": "Clasifica municipios para análisis de heterogeneidad."},

    # ── 🟢 Prioridad MEDIA ──
    
    # Rec 9: Variable de "dosis" acumulada de tratamiento
    {"prioridad": "🟢 Media", "categoria": "Feature",
     "variables": "alcaldesa_acumulado (NUEVA)",
     "transformacion": "Nº trimestres acumulados con alcaldesa hasta t",
     "razon": "Permite evaluar si el efecto se acumula con el tiempo de exposición."},

    # Rec 10: Transformación logarítmica para outcomes muy asimétricos
    {"prioridad": "🟢 Media", "categoria": "Escala",
     "variables": "outcomes per cápita",
     "transformacion": "Evaluar asinh(x) o log(1+x) si distribución es muy asimétrica",
     "razon": "Estabiliza varianza. Interpretación como semi-elasticidades."},

    # ── IDs y merges ──
    
    # Rec 11: Estandarizar la clave de municipio para merges con INEGI
    {"prioridad": "🟡 Alta", "categoria": "IDs",
     "variables": "cve_mun, cve_ent, cve_mun3, cvegeo_mun",
     "transformacion": "Usar cvegeo_mun (5 dígitos, texto) para merges con INEGI/CONAPO",
     "razon": "Formato estándar INEGI evita errores de zero-padding."},

    # Rec 12: Imputar las 2 observaciones con tipo_pob faltante
    {"prioridad": "🟢 Media", "categoria": "Imputación",
     "variables": "tipo_pob (2 NULLs)",
     "transformacion": "Asignar categoría por rango de población o excluir 2 obs",
     "razon": "Impacto mínimo. Evita perder filas en modelos con efectos fijos."},
])

# Guardar recomendaciones como CSV para referencia
recs.to_csv(OUT / "F_recomendaciones.csv", index=False)

# Mostrar tabla completa en el notebook
recs

---
## Resumen ejecutivo del EDA

### Hallazgos clave

1. **Panel limpio:** 2,471 municipios × 17 trimestres, 100% balanceado, sin duplicados.
2. **Tratamiento:** ~22% de municipios-trimestre tienen alcaldesa. 894 municipios son *switchers* (cambian de tratamiento en el periodo).
3. **Variación geográfica fuerte:** desde 51% (Baja California) hasta ~10% (EDO. MÉX.) de alcaldesas.
4. **Brecha de género visible:** en casi todos los productos, los hombres tienen más contratos per cápita que las mujeres.
5. **Correlación cruda tratamiento-outcomes:** descriptivamente, los municipios con alcaldesa tienden a ser más pequeños y con menores niveles per cápita de inclusión → sesgo de selección.
6. **Normalización per cápita es CRÍTICA:** correlación población-outcomes ~ 0.70.
7. **47 columnas con NULLs**, pero la mayoría son estructurales (saldoprom_*) o por diseño (rezagos).

### Variables para el modelo causal

| Rol | Variables |
|---|---|
| **Y (outcome)** | `ncont_*_m / pob_adulta_m` o ratio M/H |
| **T (tratamiento)** | `alcaldesa_final` |
| **FE municipio** | `cve_mun` (absorbe todo lo time-invariant) |
| **FE tiempo** | `periodo_trimestre` (absorbe shocks nacionales) |
| **Controles** | `log(pob_adulta_m)`, `tipo_pob` (si varía) |
| **Robustez** | `alcaldesa_excl_trans`, winsorización, logs |
| **Event study** | `alcaldesa_final_l1/l2/l3` + `alcaldesa_final_f1/f2/f3` |

### Próximos pasos

1. Crear variables per cápita y ratio M/H
2. Estimar modelo de efectos fijos (TWFE) con `linearmodels` o `pyfixest`
3. Event study con rezagos/adelantos para validar tendencias paralelas
4. Análisis de heterogeneidad (por región, tipo_pob, tamaño)
5. Robustez: excluir transiciones, winsorizar, diferentes medidas de outcome